In [1]:
from functions import *

In [2]:
a_vec = [763, 679, 397, 61, 697, 373, 
         289, 257, 625, 41, 193, 449]

In [3]:
G = generate_g(a_vec)
b_mat = Matrix(b_vec)
all_indices = [i for i in range(l_h**2)]
gb_indices = [3, 8]
ga_indices = [i for i in all_indices if i not in gb_indices]
Ga = G.extract(ga_indices, list(range(G.cols)))
Gb = G.extract(gb_indices, list(range(G.cols)))

NameError: name 'generate_g' is not defined

In [ ]:

basis_a = solve_modular_kernel(Ga, P)
V = Matrix.hstack(*basis_a)
print(f"V shape: {V.shape}")

V shape: (12, 12)


In [ ]:

# 2. Z3 で V * c = b を解く
coefficients = solve_decomposition_z3(V, b_mat, P)

if coefficients is None:
    print("解が見つかりませんでした。(b は V の格子に含まれていません)")
else:
    print(f"c = {coefficients.T}") # 横ベクトルで表示
    
    # # 検算
    # check_b = (V * coefficients).applyfunc(lambda x: x % P)
    # if check_b == b_mat:
    #     print("検算成功: V * c ≡ b (mod P)")
    # else:
    #     print("検算失敗")

c = Matrix([[51, 17, 37, 82, 2, 44, 8, 4, 1, 9, 0, 2]])


In [ ]:
cycles = generate_cycles(6)
h_x, h_z = generate_h_xz()
constraints = generate_constraints(cycles, a_vec, h_x, h_z)

In [ ]:
# 全ての禁止ベクトル（法ベクトル）を個別にリスト化する
unique_forbidden_vectors = []
seen_vectors = set()

# 1. 条件B (潜在部の非可換性) からの制約 r_i
for i in range(Gb.rows):
    c_prime = (Gb.row(i) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T) # 列ベクトルとして保存
        seen_vectors.add(c_tuple)

# 2. 条件C (短いサイクルの回避) からの制約 c_prime
for c in constraints:
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T)
        seen_vectors.add(c_tuple)

print(f"個別に回避すべき禁止制約（超平面）の数: {len(unique_forbidden_vectors)}")

個別に回避すべき禁止制約（超平面）の数: 312


In [ ]:
# --- Z_256 成分も動かす一般解生成の考え方 ---
def generate_full_space_solutions(x0, forbidden_vectors, p, count=10):
    x0_3 = x0.applyfunc(lambda x: x % 3)
    x0_256 = x0.applyfunc(lambda x: x % 256)
    da = x0.rows
    found_x = []
    seen_b = set() # 重複チェック用
    
    # 探索の組み合わせを増やすために random や広範な探索を検討
    # ここでは確実に異なる b を探すために seen_b を使用
    for d3 in product([0, 1, 2], repeat=da):
        for d256 in product([0, 1, 128], repeat=da):
            x3_cand = (x0_3 + Matrix(d3)).applyfunc(lambda x: x % 3)
            x256_cand = (x0_256 + Matrix(d256)).applyfunc(lambda x: x % 256)
            x_gen = crt_combine_vectors(x3_cand, x256_cand)

            # 先に b を計算して重複を確認
            b_cand = (V * x_gen).applyfunc(lambda x: x % p)
            b_tuple = tuple(b_cand)
            
            if b_tuple not in seen_b:
                if is_in_general_solution_strict(x_gen, V, forbidden_vectors, a_vec, p):
                    found_x.append(x_gen)
                    seen_b.add(b_tuple)
                    print(f"異なる解を発見 ({len(found_x)}/{count})")
            
            if len(found_x) >= count:
                return found_x
    return found_x

found_x = generate_full_space_solutions(coefficients, unique_forbidden_vectors, P)

異なる解を発見 (1/10)
異なる解を発見 (2/10)
異なる解を発見 (3/10)
異なる解を発見 (4/10)


In [ ]:
# 生成された一般解の一つを b_vec に戻して確認
true_list = []
false_list = []
if found_x:
    for b_vec in found_x:
        if check(a_vec, b_vec) == False:
            false_list.append(tuple(b_vec))
        else:
            true_list.append(tuple(b_vec))

In [1]:
for b_vec in false_list:
    # 失敗した候補 b_vec に対して
    # 1. 条件A (直交性) の確認
    print(commute_matrix(a_vec, b_vec)) 

    # 2. 条件C (ガース) の確認
    functions = generate_functions(generate_cycles(6), a_vec, b_vec, h_x, h_z)
    for f in functions:
        if is_closed(f): print(f"Closed cycle found: {f}")

NameError: name 'false_list' is not defined

In [ ]:
set(true_list)

{(435, 69, 330, 18, 612, 246, 496, 640, 200, 524, 672, 672)}